In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
api_key = os.getenv('AZURE_OPENAI_API_KEY')
vector_store_id = os.getenv('AZURE_VECTOR_STORE_ID')
endpoint

'team1-openai-sc.openai.azure.com'

### Key Authentication

In [2]:
import requests

url = f'https://{endpoint}/openai/assistants?api-version=2024-05-01-preview'
result = requests.post(
    url, 
    headers={
        'Authorization': 'Bearer YOUR_ACCESS_TOKEN',
        'api-key': api_key,
        'Content-Type': 'application/json'
    }, 
    json={
        "instructions": "시스템 지시사항이 여기에 들어갑니다.",
        "name": "도우미 이름",
        "tools": [
            {
            "type": "file_search"
            }
        ],
        "model": "gpt-4o",
        "tool_resources": {
            "file_search": {
            "vector_store_ids": [
                vector_store_id
            ]
            }
        },
        "temperature": 1,
        "top_p": 1
    }
)
result

<Response [200]>

In [3]:
result.json()

{'id': 'asst_pyrOWbCC8KGhObjjhr2UXg6H',
 'object': 'assistant',
 'created_at': 1744097186,
 'name': '도우미 이름',
 'description': None,
 'model': 'gpt-4o',
 'instructions': '시스템 지시사항이 여기에 들어갑니다.',
 'tools': [{'type': 'file_search'}],
 'top_p': 1.0,
 'temperature': 1.0,
 'tool_resources': {'file_search': {'vector_store_ids': ['vs_tfjfiRm96tCMQzueruozXHwp']}},
 'metadata': {},
 'response_format': 'auto'}

In [99]:
# assistant_id = result.json()['id']
assistant_id = "asst_oK0SfknufRk02g2JuE3xiwK9"

### Create a thread

In [100]:
url = f'https://{endpoint}/openai/threads?api-version=2024-05-01-preview'

result = requests.post(
    url, 
    headers={
        'api-key': api_key,
        'Content-Type': 'application/json|'
    }
)
result

<Response [200]>

In [101]:
result.json()

{'id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
 'object': 'thread',
 'created_at': 1744804638,
 'metadata': {},
 'tool_resources': {}}

In [102]:
thread_id = result.json()['id']

### Add a user question to the thread

In [123]:
url = f'https://{endpoint}/openai/threads/{thread_id}/messages?api-version=2024-05-01-preview'

result = requests.post(
    url, 
    headers={
        'api-key': api_key,
        'Content-Type': 'application/json'
    },
    json={
        'role': 'user',
        'content': '서울 강남지역 채용공고 찾아줘.'
    }
)
result

<Response [200]>

In [124]:
result.json()

{'id': 'msg_hPxwj5FxnZLwHhftF4Yny0VE',
 'object': 'thread.message',
 'created_at': 1744804804,
 'assistant_id': None,
 'thread_id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
 'run_id': None,
 'role': 'user',
 'content': [{'type': 'text',
   'text': {'value': '서울 강남지역 채용공고 찾아줘.', 'annotations': []}}],
 'attachments': [],
 'metadata': {}}

### Run the thread

In [125]:
url = f'https://{endpoint}/openai/threads/{thread_id}/runs?api-version=2024-05-01-preview'

result = requests.post(
    url, 
    headers={
        'api-key': api_key,
        'Content-Type': 'application/json'
    },
    json={
        'assistant_id': assistant_id
    }
)
result

<Response [200]>

In [126]:
result.json()

{'id': 'run_Sv12G4b9d07WExtDtiGxdo2Y',
 'object': 'thread.run',
 'created_at': 1744804806,
 'assistant_id': 'asst_oK0SfknufRk02g2JuE3xiwK9',
 'thread_id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
 'status': 'queued',
 'started_at': None,
 'expires_at': 1744805406,
 'cancelled_at': None,
 'failed_at': None,
 'completed_at': None,
 'required_action': None,
 'last_error': None,
 'model': 'gpt-4o',
 'instructions': '🎯 역할 (Role)\n당신의 역할은 사용자가 제공한 조건이나 이력서를 기반으로 적합한 채용공고를 매칭하거나 추천하는 것입니다.\n사용자가 원하는 요구사항과 목표를 철저히 이해한 뒤, 해당하는 채용공고를 논리적으로 분석해 추천하세요.\n\n🧩 진행 순서\n1.사용자 친화적인 요약\n2.가장 높은 매칭률 기준 정렬\n3.최대 5개 공고 추천\n4.깔끔한 포맷 출력\n5.매칭 퍼센트 표시\n\n📢 데이터 정확성 보장\n데이터의 정확성을 위해 벡터 저장소의 정보만 신뢰해서 채용공고를 매칭합니다.\n\n✅ 출력 순서\n📜 사용자 이력서 요약 (사용자의 이력서가 첨부된 경우만 출력합니다.)\n\n**이력서 요약**:[사용자가 첨부한 이력서 바탕으로 짧게 요약]\n**경력 요약** :[예: 신입이면 신입이라고 출력해주고 경력자면 몇 년 경력이 있는지 반드시 출력]\n**사용 기술 스택**:[예시 : Python, Django, REST API 경험]\n**관심 분야 및 희망하는 업무**:[예시: 데이터 분야, 백엔드 개발자, 프론트엔드 개발자 등]\n\n**➡️매칭 전략** :\n[요약 후 한 줄로 매칭 전략 간단 설명 ("이런 경력을 바탕으로 적합한

In [127]:
run_id = result.json()['id']

### Get the status of the run

In [128]:
url = f'https://{endpoint}/openai/threads/{thread_id}/runs/{run_id}?api-version=2024-05-01-preview'

result = requests.get(
    url, 
    headers={
        'api-key': api_key,
    }
)
result

<Response [200]>

In [129]:
result.json()

{'id': 'run_Sv12G4b9d07WExtDtiGxdo2Y',
 'object': 'thread.run',
 'created_at': 1744804806,
 'assistant_id': 'asst_oK0SfknufRk02g2JuE3xiwK9',
 'thread_id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
 'status': 'in_progress',
 'started_at': 1744804806,
 'expires_at': 1744805406,
 'cancelled_at': None,
 'failed_at': None,
 'completed_at': None,
 'required_action': None,
 'last_error': None,
 'model': 'gpt-4o',
 'instructions': '🎯 역할 (Role)\n당신의 역할은 사용자가 제공한 조건이나 이력서를 기반으로 적합한 채용공고를 매칭하거나 추천하는 것입니다.\n사용자가 원하는 요구사항과 목표를 철저히 이해한 뒤, 해당하는 채용공고를 논리적으로 분석해 추천하세요.\n\n🧩 진행 순서\n1.사용자 친화적인 요약\n2.가장 높은 매칭률 기준 정렬\n3.최대 5개 공고 추천\n4.깔끔한 포맷 출력\n5.매칭 퍼센트 표시\n\n📢 데이터 정확성 보장\n데이터의 정확성을 위해 벡터 저장소의 정보만 신뢰해서 채용공고를 매칭합니다.\n\n✅ 출력 순서\n📜 사용자 이력서 요약 (사용자의 이력서가 첨부된 경우만 출력합니다.)\n\n**이력서 요약**:[사용자가 첨부한 이력서 바탕으로 짧게 요약]\n**경력 요약** :[예: 신입이면 신입이라고 출력해주고 경력자면 몇 년 경력이 있는지 반드시 출력]\n**사용 기술 스택**:[예시 : Python, Django, REST API 경험]\n**관심 분야 및 희망하는 업무**:[예시: 데이터 분야, 백엔드 개발자, 프론트엔드 개발자 등]\n\n**➡️매칭 전략** :\n[요약 후 한 줄로 매칭 전략 간단 설명 ("이런 경

### See the Assistant response

In [130]:
url = f'https://{endpoint}/openai/threads/{thread_id}/messages?api-version=2024-05-01-preview'

result = requests.get(
    url, 
    headers={
        'api-key': api_key,
        'Content-Type': 'application/json'
    }
)
result

<Response [200]>

In [131]:
result.json()

{'object': 'list',
 'data': [{'id': 'msg_hPxwj5FxnZLwHhftF4Yny0VE',
   'object': 'thread.message',
   'created_at': 1744804804,
   'assistant_id': None,
   'thread_id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
   'run_id': None,
   'role': 'user',
   'content': [{'type': 'text',
     'text': {'value': '서울 강남지역 채용공고 찾아줘.', 'annotations': []}}],
   'attachments': [],
   'metadata': {}},
  {'id': 'msg_Ri5bDjSkO2zSIHJnwTl0BwvA',
   'object': 'thread.message',
   'created_at': 1744804754,
   'assistant_id': 'asst_oK0SfknufRk02g2JuE3xiwK9',
   'thread_id': 'thread_bzH0Le8RynY9dHJY1xr1OiQn',
   'run_id': 'run_EPpC8kArniR5I5CRVEUYyhev',
   'role': 'assistant',
   'content': [{'type': 'text',
     'text': {'value': '## 💡추천 공고 1  \n**매칭 퍼센트** : 95% 높은 매칭 🟢  \n**회사 이름** : 오케스트로  \n**채용 직군** : Architect  \n**요구 기술 스택** : [명시 없음]  \n**위치** : 서울 영등포구 여의대로 108, 파크원 NH금융타워 43층  \n**조직 문화** : 자유출근, 점심 식대 지원, 개인 장비 지급  \n**지원 링크** : [상세 공고 바로가기](https://www.wanted.co.kr/wd/272685)【8:0†source】  \n\n#### **🔎 매칭